# YOLO11n Indoor Accessibility — Cloud Training
Run each cell **top to bottom** in order.

> **Before you start**: set the runtime to **GPU**  
> Runtime → Change runtime type → T4 GPU (free) or A100 (Colab Pro)

### What this notebook does
1. Downloads your Roboflow dataset  
2. Fetches COCO 2017 annotations and builds the merged indoor dataset  
3. Trains YOLO11n (logic identical to `train_yolo11n.py`)  
4. Downloads `best.pt` to your machine  

The trained `best.pt` can then be deployed to **Jetson Nano** directly (PyTorch) or converted to TensorRT on the Jetson for faster inference.

## Cell 1 — Install dependencies

In [ ]:
!pip install -q ultralytics roboflow
import torch
print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Cell 2 — Configuration
**Edit `ROBOFLOW_API_KEY` before running.**

In [ ]:
from pathlib import Path

# ── Roboflow credentials ──────────────────────────────────────────────────────
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"   # <-- paste your Private API Key here

# ── Paths ─────────────────────────────────────────────────────────────────────
# Mirrors the layout of the original scripts where PROJECT_ROOT is the repo root.
# Here we use /content as the root so everything persists within the session.
PROJECT_ROOT = Path("/content")
DATASET_ROOT = PROJECT_ROOT / "dataset"
DATASET_ROOT.mkdir(parents=True, exist_ok=True)

CUSTOM_DIR   = DATASET_ROOT / "my-second-project-person-chair-add-chair.v2i.yolov8"
OUTPUT_DIR   = DATASET_ROOT / "indoor_coco_merged"
CACHE_DIR    = DATASET_ROOT / "coco_cache"
WEIGHTS_PATH = PROJECT_ROOT / "yolo11n.pt"

# ── Training hyper-parameters (identical defaults to train_yolo11n.py) ────────
EPOCHS   = 100
IMGSZ    = 640
BATCH    = 16     # 16 is safe for T4; raise to 32-64 on A100
WORKERS  = 4
PATIENCE = 20
RUN_NAME = "yolo11n_indoor_accessibility"

print("Paths:")
print(f"  Custom dataset : {CUSTOM_DIR}")
print(f"  Merged output  : {OUTPUT_DIR}")
print(f"  Weights        : {WEIGHTS_PATH}")
print(f"  Training runs  : {PROJECT_ROOT / 'runs' / 'detect' / RUN_NAME}")

## Cell 3 — Download Roboflow dataset
Logic mirrors `download_roboflow_dataset.py`.

In [ ]:
# Mirrors download_roboflow_dataset.py
WORKSPACE = "test-workspace-dwuie"
PROJECT   = "my-second-project-person-chair-add-chair"
VERSION   = 2
FORMAT    = "yolov8"

if CUSTOM_DIR.exists():
    print(f"Dataset already exists at {CUSTOM_DIR} — skipping download.")
else:
    from roboflow import Roboflow
    rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(WORKSPACE).project(PROJECT)
    version = project.version(VERSION)
    dataset = version.download(FORMAT, location=str(CUSTOM_DIR), overwrite=False)
    print(f"Downloaded to: {dataset.location}")

## Cell 4 — Download YOLO11n pretrained weights

In [ ]:
import urllib.request

if not WEIGHTS_PATH.exists():
    print("Downloading yolo11n.pt ...")
    urllib.request.urlretrieve(
        "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt",
        str(WEIGHTS_PATH)
    )
    print(f"Saved to {WEIGHTS_PATH}")
else:
    print(f"Weights already at {WEIGHTS_PATH}")

## Cell 5 — Build merged indoor-COCO dataset
Logic mirrors `build_indoor_coco_dataset.py` with the same defaults.

In [ ]:
# ── Mirrors build_indoor_coco_dataset.py ──────────────────────────────────────
import json, os, random, shutil, urllib.request as _urllib, zipfile
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

ANNOTATIONS_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
IMAGE_SUFFIXES  = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

CUSTOM_CLASSES = [
    "chair", "cleaning_cart", "handrail", "person", "pillars",
    "ramp", "squat-toilet", "stairs", "urinals", "vending_machine", "wet_floor_sign",
]

COCO_CLASS_MAP = {
    "person": "person",       "bicycle": "bicycle",     "chair": "chair",
    "bench": "bench",         "cat": "cat",             "dog": "dog",
    "backpack": "backpack",   "umbrella": "umbrella",   "handbag": "handbag",
    "suitcase": "suitcase",   "bottle": "bottle",       "bed": "bed",
    "dining table": "table",  "couch": "sofa",          "toilet": "toilet",
    "potted plant": "potted_plant", "sink": "sink",     "refrigerator": "refrigerator",
}

INDOOR_ANCHORS = {
    "chair", "bed", "dining table", "couch", "toilet",
    "potted plant", "sink", "refrigerator",
}

# Same defaults as build_indoor_coco_dataset.py
TRAIN_PER_CLASS  = 500
VAL_PER_CLASS    = 100
MAX_TRAIN_IMAGES = 5000
MAX_VAL_IMAGES   = 1000
BUILD_WORKERS    = 8
SEED             = 42


def _download(url, destination):
    if destination.is_file() and destination.stat().st_size:
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    tmp = destination.with_suffix(destination.suffix + ".part")
    req = _urllib.Request(url, headers={"User-Agent": "indoor-coco-builder/1.0"})
    try:
        with _urllib.urlopen(req, timeout=60) as r, tmp.open("wb") as f:
            shutil.copyfileobj(r, f)
        tmp.replace(destination)
    finally:
        tmp.unlink(missing_ok=True)


def _ensure_annotations(cache_root):
    archive = cache_root / "annotations_trainval2017.zip"
    print("Ensuring COCO annotations ...")
    _download(ANNOTATIONS_URL, archive)
    ann_dir = cache_root / "annotations"
    outputs = {}
    with zipfile.ZipFile(archive) as bundle:
        for split in ("train2017", "val2017"):
            dest = ann_dir / f"instances_{split}.json"
            outputs[split] = dest
            if not dest.is_file():
                dest.parent.mkdir(parents=True, exist_ok=True)
                with bundle.open(f"annotations/instances_{split}.json") as src, dest.open("wb") as out:
                    shutil.copyfileobj(src, out)
    return outputs


def _select_images(coco, quota, max_images, seed):
    category_name = {c["id"]: c["name"] for c in coco["categories"]}
    target_ids = {cid for cid, n in category_name.items() if n in COCO_CLASS_MAP}
    anchor_ids  = {cid for cid, n in category_name.items() if n in INDOOR_ANCHORS}
    ann_by_img  = defaultdict(list)
    cat_by_img  = defaultdict(set)
    for a in coco["annotations"]:
        if a.get("iscrowd", 0):
            continue
        ann_by_img[a["image_id"]].append(a)
        cat_by_img[a["image_id"]].add(a["category_id"])
    eligible = {iid for iid, cats in cat_by_img.items()
                if cats & target_ids and cats & anchor_ids}
    cands = {cid: [iid for iid in eligible if cid in cat_by_img[iid]]
             for cid in target_ids}
    rng = random.Random(seed)
    for v in cands.values():
        rng.shuffle(v)
    selected, counts = set(), Counter()
    for cid in sorted(target_ids, key=lambda c: len(cands[c])):
        for iid in cands[cid]:
            if counts[cid] >= quota or iid in selected:
                continue
            if len(selected) >= max_images:
                break
            selected.add(iid)
            for pid in cat_by_img[iid] & target_ids:
                counts[pid] += 1
    imgs = {img["id"]: img for img in coco["images"]}
    return sorted(selected), ann_by_img, imgs, category_name, counts


def _link_or_copy(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def _validate_custom_dataset(custom_root):
    for source_split in ("train", "valid", "test"):
        for folder in ("images", "labels"):
            path = custom_root / source_split / folder
            if not path.is_dir():
                raise FileNotFoundError(f"Missing custom dataset directory: {path}")


def _add_custom(custom_root, output_root):
    counts = Counter()
    split_map = {"train": "train", "valid": "val", "test": "test"}
    for source_split, output_split in split_map.items():
        source_images = custom_root / source_split / "images"
        for image_path in sorted(source_images.iterdir()):
            if image_path.suffix.lower() not in IMAGE_SUFFIXES:
                continue
            output_stem = f"custom__{image_path.stem}"
            output_image = output_root / "images" / output_split / (
                output_stem + image_path.suffix.lower())
            source_label = custom_root / source_split / "labels" / f"{image_path.stem}.txt"
            output_label = output_root / "labels" / output_split / f"{output_stem}.txt"
            if not source_label.is_file():
                raise FileNotFoundError(f"Image has no matching label: {image_path}")
            for line_number, line in enumerate(source_label.read_text().splitlines(), 1):
                if not line.strip():
                    continue
                class_id = int(line.split()[0])
                if class_id < 0 or class_id >= len(CUSTOM_CLASSES):
                    raise ValueError(f"Invalid class {class_id} at {source_label}:{line_number}")
                counts[CUSTOM_CLASSES[class_id]] += 1
            _link_or_copy(image_path, output_image)
            _link_or_copy(source_label, output_label)
    return counts


def _coco_box_to_yolo(annotation, width, height):
    x, y, box_width, box_height = annotation["bbox"]
    left   = max(0.0, min(float(width),  float(x)))
    top    = max(0.0, min(float(height), float(y)))
    right  = max(0.0, min(float(width),  float(x + box_width)))
    bottom = max(0.0, min(float(height), float(y + box_height)))
    if right <= left or bottom <= top:
        return None
    return (
        ((left + right)  / 2.0) / width,
        ((top  + bottom) / 2.0) / height,
        (right - left) / width,
        (bottom - top) / height,
    )


def _add_coco_split(split, out_split, ids, ann_by_img, imgs, cat_name, cache_root, out_root, classes):
    final_class_id = {name: index for index, name in enumerate(classes)}
    counts = Counter()
    for image_id in ids:
        image = imgs[image_id]
        output_stem = f"coco2017__{Path(image['file_name']).stem}"
        cached_image = cache_root / "images" / split / image["file_name"]
        output_image = out_root / "images" / out_split / (
            output_stem + Path(image["file_name"]).suffix.lower())
        output_label = out_root / "labels" / out_split / f"{output_stem}.txt"
        lines = []
        for annotation in ann_by_img[image_id]:
            coco_name = cat_name[annotation["category_id"]]
            if coco_name not in COCO_CLASS_MAP:
                continue
            box = _coco_box_to_yolo(annotation, image["width"], image["height"])
            if box is None:
                continue
            final_name = COCO_CLASS_MAP[coco_name]
            lines.append(f"{final_class_id[final_name]} " + " ".join(f"{v:.6f}" for v in box))
            counts[final_name] += 1
        if not lines:
            raise ValueError(f"Selected COCO image {image_id} produced no labels")
        _link_or_copy(cached_image, output_image)
        output_label.parent.mkdir(parents=True, exist_ok=True)
        output_label.write_text("\n".join(lines) + "\n")
    return counts


# ── Run the build ─────────────────────────────────────────────────────────────
if OUTPUT_DIR.exists():
    print(f"Merged dataset already exists at {OUTPUT_DIR} — skipping.")
    print("Delete the folder and re-run this cell to rebuild.")
else:
    _validate_custom_dataset(CUSTOM_DIR)
    annotation_paths = _ensure_annotations(CACHE_DIR)

    selections = {}
    for i, split in enumerate(("train2017", "val2017")):
        print(f"Reading {annotation_paths[split]} ...")
        with annotation_paths[split].open() as f:
            coco = json.load(f)
        quota = TRAIN_PER_CLASS if split == "train2017" else VAL_PER_CLASS
        maxim = MAX_TRAIN_IMAGES if split == "train2017" else MAX_VAL_IMAGES
        sel = _select_images(coco, quota, maxim, SEED + i)
        selections[split] = sel
        ids, _, _, category_name, counts = sel
        print(f"{split}: selected {len(ids)} indoor-proxy images")
        for cid, cnt in sorted(counts.items(), key=lambda item: category_name[item[0]]):
            print(f"  {category_name[cid]}: {cnt} images")

    # Download COCO images
    jobs = []
    for split, (ids, _, imgs_by_id, _, _) in selections.items():
        idir = CACHE_DIR / "images" / split
        for iid in ids:
            img = imgs_by_id[iid]
            jobs.append((img["coco_url"], idir / img["file_name"]))
    print(f"\nDownloading/verifying {len(jobs)} COCO images (cached on re-run) ...")
    failures, done = [], 0
    with ThreadPoolExecutor(max_workers=BUILD_WORKERS) as ex:
        futs = {ex.submit(_download, url, dst): url for url, dst in jobs}
        for fut in as_completed(futs):
            try:
                fut.result()
            except Exception as e:
                failures.append(f"{futs[fut]}: {e}")
            done += 1
            if done % 250 == 0 or done == len(jobs):
                print(f"  Downloaded/verified {done}/{len(jobs)} COCO images", flush=True)
    if failures:
        raise RuntimeError(f"Failed downloads:\n" + "\n".join(failures[:10]))

    # Build output directory structure
    classes = CUSTOM_CLASSES + [n for n in COCO_CLASS_MAP.values() if n not in CUSTOM_CLASSES]
    for folder in ("images", "labels"):
        for s in ("train", "val", "test"):
            (OUTPUT_DIR / folder / s).mkdir(parents=True, exist_ok=True)

    print("\nMerging custom dataset ...")
    custom_counts = _add_custom(CUSTOM_DIR, OUTPUT_DIR)
    coco_counts   = Counter()
    for split, out_split in (("train2017", "train"), ("val2017", "val")):
        ids, ann_by_img, imgs, category_name, _ = selections[split]
        coco_counts.update(
            _add_coco_split(split, out_split, ids, ann_by_img, imgs,
                            category_name, CACHE_DIR, OUTPUT_DIR, classes)
        )

    # Write data.yaml  (same format as original script)
    yaml_lines = [
        f"path: {OUTPUT_DIR.resolve()}",
        "train: images/train",
        "val:   images/val",
        "test:  images/test",
        "",
        f"nc: {len(classes)}",
        "names:",
    ]
    yaml_lines += [f"  {i}: {n}" for i, n in enumerate(classes)]
    (OUTPUT_DIR / "data.yaml").write_text("\n".join(yaml_lines) + "\n")

    img_counts = {s: sum(p.suffix.lower() in IMAGE_SUFFIXES
                         for p in (OUTPUT_DIR / "images" / s).iterdir())
                  for s in ("train", "val", "test")}
    print(f"\nBuilt merged dataset at {OUTPUT_DIR}")
    print(f"Images by split: {img_counts}")
    print(f"Custom boxes: {sum(custom_counts.values())} | COCO boxes: {sum(coco_counts.values())}")

## Cell 6 — Train YOLO11n
Logic mirrors `train_yolo11n.py` exactly.

In [ ]:
# ── Mirrors train_yolo11n.py ──────────────────────────────────────────────────
import torch
from ultralytics import YOLO

DATA_YAML = OUTPUT_DIR / "data.yaml"

if not DATA_YAML.is_file():
    raise FileNotFoundError(
        f"Dataset config not found: {DATA_YAML}\nRun Cell 5 first."
    )
if not WEIGHTS_PATH.is_file():
    raise FileNotFoundError(f"YOLO11n weights not found: {WEIGHTS_PATH}\nRun Cell 4 first.")

device = "0" if torch.cuda.is_available() else "cpu"
print(f"Training on device={device}")
if device == "cpu":
    print("WARNING: No GPU detected. Training will be very slow.")
    print("Go to Runtime -> Change runtime type -> T4 GPU.")

model = YOLO(str(WEIGHTS_PATH), task="detect")
model.train(
    data          = str(DATA_YAML),
    epochs        = EPOCHS,
    imgsz         = IMGSZ,
    batch         = BATCH,
    device        = device,
    workers       = WORKERS,
    patience      = PATIENCE,
    pretrained    = True,
    amp           = device != "cpu",
    project       = str(PROJECT_ROOT / "runs" / "detect"),
    name          = RUN_NAME,
    exist_ok      = False,
    seed          = 42,
    deterministic = True,
    plots         = True,
    val           = True,
    save          = True,
)

BEST_PT = PROJECT_ROOT / "runs" / "detect" / RUN_NAME / "weights" / "best.pt"
print(f"\nTraining complete!")
print(f"Best weights: {BEST_PT}")

## Cell 7 — Download best.pt
Downloads the trained `best.pt` weights to your local machine.

The `.pt` file can be used directly on **Jetson Nano** with `ultralytics` (PyTorch inference),  
or later converted to TensorRT on the Jetson for faster inference.

In [ ]:
from google.colab import files
from pathlib import Path

BEST_PT   = PROJECT_ROOT / "runs" / "detect" / RUN_NAME / "weights" / "best.pt"
LAST_PT   = PROJECT_ROOT / "runs" / "detect" / RUN_NAME / "weights" / "last.pt"

for pt in [BEST_PT, LAST_PT]:
    if pt.exists():
        print(f"Downloading {pt.name} ...")
        files.download(str(pt))
    else:
        print(f"WARNING: {pt.name} not found — has training completed?")